# Saint-Sauveur

Deux sondes : **CTD** (Diver autonome : niveau, conductivité, température), **TROLL** (Aqua TROLL : conductivité, température, turbidité, O2, chlorophylle).


## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ouysse
from ouysse import *

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, "Saint Sauveur_consolide_OLD.xlsx")
UTC_CTD_PATH     = os.path.join(BASE, "UTC_CTD.xlsx")
UTC_TROLL_PATH   = os.path.join(BASE, "UTC_Troll.xlsx")
PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements_Niveau.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_Conducti.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, "Saint_Sauveur_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Saint_Sauveur_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Saint-Sauveur"
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. CTD : lecture, UTC et compensation barométrique

`lire_CTD` encaisse les pièges du format Diver (en-tête à une ligne variable, pied
`END OF DATA`, virgules décimales, mS/cm ou µS/cm). La table UTC nomme les fichiers
exactement, sinon la campagne est ignorée et le message la nomme.

In [ ]:
metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom, CTD_PATH, PAS), nom, metadata)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
    morceaux.append(m[["Date/time", "Niveau_(cm)", "Cond_(µS/cm)", "Temp_(°C)"]])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements en UTC.")

## 4. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes récentes sont la **même sonde CTD**, séparées
par un trou d'exploitation : le décalage est mesuré à la jonction et appliqué aux
campagnes, pour que la chronique soit continue.

In [ ]:
#: L'ancien consolidé n'emploie pas les mêmes noms de colonnes du notebook.
RENOMMAGE_OLD = {
    "Niveau_(cm)": "Niveau_CTD_(cm)",
}
olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = (olddata_df.rename(columns=RENOMMAGE_OLD).dropna(subset=["DATE"])
              .sort_values("DATE"))

merge_ctd_df = raccorder_campagnes(olddata_df, merge_ctd_df, [
    ("Niveau", "Niveau_CTD_(cm)", "Niveau_(cm)", "cm"),
    ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")])

## 5. TROLL : exports VuSitu

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux = []
for nom in fichiers(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = en_utc(lire_VuSitu(nom, VUSITU_PATH, PAS), nom, metadata_troll,
                                col_date="DATE")
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    morceaux.append(df_v)
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(df_v)} lignes)")

merge_troll_df = (pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
                  .sort_values("DATE", kind="stable"))
print(f"\n{len(morceaux)} fichier(s), {len(merge_troll_df)} enregistrements en UTC.")

## 6. Assemblage : une colonne par sonde

In [ ]:
COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]
COLONNES_TROLL = ["Cond_Troll_(µS/cm)", "température_Troll_(°C)", "Turbidity_Troll_(NTU)",
                  "O2_Troll_(mg/l)", "O2 (%Sat)", "FluorescenceChloro_a_Troll_(RFU)",
                  "ConcentrationChloro_a_(µg/l)"]

full_data = sur_grille([
    empiler([olddata_df,
             merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                          "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                          "Temp_(°C)": "Temp _CTD(°C)"})], COLONNES_CTD),
    empiler([olddata_df, merge_troll_df], COLONNES_TROLL),
], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

#: Les sondes de chaque grandeur : {grandeur: {sonde: colonne}}.
SONDES = {
    "Niveau_(cm)":        {"CTD": "Niveau_CTD_(cm)"},
    "Conductivité":       {"TROLL": "Cond_Troll_(µS/cm)", "CTD": "Cond_CTD_(µS/cm)"},
    "Température":        {"TROLL": "température_Troll_(°C)", "CTD": "Temp _CTD(°C)"},
    "Turbidité_(NTU)":    {"TROLL": "Turbidity_Troll_(NTU)"},
    "O2_(mg/l)":          {"TROLL": "O2_Troll_(mg/l)"},
    "Chlorophylle_(RFU)": {"TROLL": "FluorescenceChloro_a_Troll_(RFU)"},
}
PARAMETRES = list(SONDES)

#: Ordre de préférence automatique : à chaque pas, la première sonde qui mesure.
ORDRE = ["TROLL", "CTD"]

BRUT = full_data.copy()
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 7. Corrections capteur

Les deux seules corrections qui portent sur une **sonde**, avant toute fusion.

`VOIES_ECARTEES` met des mesures à l'écart : la voie est retirée avant la fusion, donc
une autre sonde prend le relais si elle mesure ; s'il n'y en a pas, la lacune reste et
l'interpolation ne comblera pas plus de 12 h.

`CALAGES_SONDE` déplace une sonde entière, ou son passé, ou son avenir.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart, toutes grandeurs.
VOIES_ECARTEES = [
    ("2020-06-07 14:00", "2020-06-07 14:00", "Niveau_CTD_(cm)", "pic isolé"),
    ("2021-09-09 12:00", "2021-09-09 13:00", "Niveau_CTD_(cm)", "pic isolé"),
    ("2020-07-16 15:00","2020-07-16 20:00","Cond_CTD_(µS/cm)","pic isolé")
]
#: (date d'ancrage, colonne, décalage, sens) : ajustements manuels d'une voie.
#: sens = "amont" (avant la date) | "aval" (à partir de la date) | "tout".
CALAGES_SONDE = [
    ("2020-07-16 13:00", "Cond_CTD_(µS/cm)", -25, "amont"),
]
print("Voies écartées :")
CORRIGE = ecarter(BRUT.copy(), VOIES_ECARTEES)

def voies_calees(grandeur):
    """Les séries des sondes d'une grandeur, écarts et calages appliqués."""
    series = {}
    for sonde, col in SONDES[grandeur].items():
        s = CORRIGE[col]
        for date, cible, valeur, sens in CALAGES_SONDE:
            if cible not in CORRIGE.columns:
                raise ValueError(f"CALAGES_SONDE : colonne inconnue {cible!r}")
            if cible == col:
                s = decaler(s, date, valeur, sens)
                print(f"  {col} : {valeur:+.2f} en {sens} du "
                      f"{pd.to_datetime(date):%d/%m/%Y %H:%M}")
        series[sonde] = s
    return series

## 8. Comparaison des sources

À lancer pour juger quelle sonde garder sur une période, avant d'écrire une période
imposée dans les cellules suivantes.

In [ ]:
PARAMETRE = "Conductivité"   # "Niveau_(cm)", "Conductivité", "Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"

graphe_sondes({sonde: BRUT[col] for sonde, col in SONDES[PARAMETRE].items()},
              titre=f"{PARAMETRE} : les sondes disponibles", ylab=PARAMETRE)

## 9. Niveau

La sonde est choisie automatiquement, dans l'ordre `ORDRE` déclaré à l'assemblage.
`SONDE_PRIORITAIRE_NIVEAU` sert à imposer une autre sonde sur une période précise. Les
périodes retenues sont affichées, avec le recalage appliqué à chaque changement.

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_NIVEAU = [
]
points_niveau = lire_points(PUNCTUAL_NIVEAU)

print("Calages de sonde :")
voies = voies_calees("Niveau_(cm)")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_NIVEAU), "cm")

print("Points de contrôle :")
niveau = caler(avant, points_niveau, "Hauteur (cm)")
full_data["Niveau_(cm)"], full_data["Niveau_(cm)_source"] = niveau, source
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe_sondes(voies, avant, niveau, titre="Niveau", ylab="Niveau (cm)",
              points=points_niveau, col_point="Hauteur (cm)")

## 10. Conductivité

In [ ]:
#: (début, fin, sonde imposée) : sort du choix automatique sur cette période.
SONDE_PRIORITAIRE_COND = [
]
points_cond = lire_points(PUNCTUAL_CONDUCT)

print("Calages de sonde :")
voies = voies_calees("Conductivité")
print("Périodes retenues :")
avant, source = fusionner(voies, choisir_sondes(voies, ORDRE, SONDE_PRIORITAIRE_COND), "µS/cm")

print("Points de contrôle :")
cond = caler(avant, points_cond, "Conductivité")
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe_sondes(voies, avant, cond, titre="Conductivité", ylab="Conductivité (µS/cm)",
              points=points_cond, col_point="Conductivité")

### Filtre IQR et lissage

Post-traitement appliqué **après** la fusion et le calage sur les points de contrôle :
c'est cette chronique nettoyée qui alimente `full_data` et le fichier final.

In [ ]:
FENETRE_IQR, K_IQR = "48h", 0.8   # k = 0 : pas de filtre
LISSAGE_H = 0                     # 0 = pas de lissage ; sinon médiane glissante, en heures

cond_iqr = filtre_iqr(cond, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)
full_data["Conductivité"], full_data["Conductivité_source"] = cond_iqr, source
full_data["Conductivité_Moyenne_Mobile"] = cond_iqr.rolling("6h", center=True).mean()

graphe([(cond, "avant IQR et lissage", "darkorange"),
        (cond_iqr, "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 11. Température et autres paramètres

Choix automatique, pas de calage sur points de contrôle. Les voies défaillantes ont
déjà été écartées à la cellule des corrections capteur.

In [ ]:
#: {grandeur: [(début, fin, sonde imposée)]} pour sortir du choix automatique.
EXCEPTIONS = {
}

for grandeur in ["Température", "Turbidité_(NTU)", "O2_(mg/l)", "Chlorophylle_(RFU)"]:
    print(f"{grandeur} :")
    voies = voies_calees(grandeur)
    full_data[grandeur], full_data[f"{grandeur}_source"] = fusionner(
        voies, choisir_sondes(voies, ORDRE, EXCEPTIONS.get(grandeur, [])))
    print(f"  {full_data[grandeur].notna().sum()} pas  "
          f"{full_data[f'{grandeur}_source'].value_counts().to_dict()}")

## 12. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur est
mesurée, interpolée ou manquante.

`cote = 107.611 + Niveau_(cm) / 100`. L'ancienne version écrivait `ngf - (ngf - h) / 100`,
qui se simplifie en `106.5349 + h / 100` et plaçait donc le zéro 1.0761 m trop bas.

In [ ]:
NIVEAU_NGF = 107.611       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 13. Sauvegarde et graphe de synthèse

Un paramètre par grandeur, avec son statut. Le détail capteur par capteur, et la sonde
retenue à chaque pas, restent dans le fichier consolidé écrit à l'assemblage. La version
de `ouysse-hydro` est affichée : c'est elle qui dit avec quel code la chronique a été
produite.

In [ ]:
finaux = [c for c in PARAMETRES + ["Q_()", "Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Niveau_(cm)", "Niveau (cm)", pluie=PLUIE_PATH, sortie=SORTIE_SVG,marge_jours=15)

### Contrôle des mesures et des interpolations

Un paramètre à la fois, choisi dans le menu du graphe : les pas mesurés en noir, les pas
comblés par interpolation en rouge. Les trous de plus de 12 h restent vides.

In [ ]:
graphe_statuts(full_data, finaux)